# MTG Contrail Pipeline

This notebook retrieves contrail detections for MTG from the Google Contrails API and decomposes them into individual contrails. For tracking, advection, and matching, see [goes_pipeline](goes_pipeline.ipynb).

> Before running, set your API keys in `~/.env`, as described in the README.

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np

# Import contrail-detect modules (more below)
from contrail_detect.geostationary.abc import Geostationary
from contrail_detect.models.contrail_model_base import ContrailModelBase

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)

In [ ]:
# Display valid datetime ranges for satellite & detection options
Geostationary.display_valid_datetime_ranges()
ContrailModelBase.display_detection_dates()

## Detector Set-up

Here, we use ten minutes of MTG Full Disk images.

In [ ]:
from contrail_detect.geostationary.meteosat import MeteosatDetector

In [ ]:
satellite = "MTG"
region = "F"
start_time = np.datetime64("2026-01-01T09:00:00")
end_time = np.datetime64("2026-01-01T09:10:00")

meteosat_detector = MeteosatDetector(
    meteosat_region=region,
    meteosat_satellite=satellite,
    start_time=start_time,
    end_time=end_time,
)

## Contrail Detection

We get the detection masks from the Google Contrails API (`google_api`), which is currently the only method available for MTG and requires a Google Contrails API key.

Detections are cached locally. To retrieve them again, set `force_reprocess_detections = True`.

In [ ]:
detection_method = "google_api"

force_reprocess_detections = False

In [ ]:
detections = meteosat_detector.get_detections(
    method=detection_method,
    force_reprocess=force_reprocess_detections,
)

In [ ]:
meteosat_detector.detections = detections

#### Detection Masks

This optional plot shows the detection masks for each chosen timestamp. Setting `show_domain=True` outlines the area covered by the detections. To save the figures rather than show them, pass `fig_output_dir` to `plot_detections`.

In [ ]:
plot_detections = True

timestamps_to_plot = meteosat_detector.timestamp_sequence[:1]

if plot_detections:
    figures = meteosat_detector.plot_detections(
        timestamps=timestamps_to_plot,
        show_domain=True,
    )
    plt.show()

## Detection Mask Decomposition

Next, we split the detection masks into individual contrails using one of two methods:

- `decomposer` decomposes the masks locally.
- `google_api` retrieves line segments from the Google Contrails API, without point clouds, and requires a Google Contrails API key.

The decomposer takes two settings. First, `prediction_threshold` is the detection threshold, i.e., the detection score below which pixels are dropped. If it is left as `None`, the default for the detection source is used (0.40 for the Google Contrails API). Second, `contrail_stitching` sets whether fragments split from overlapping contrails are joined back into whole contrails.

The decomposer writes `contrail_summary`, with one row per contrail, and `contrail_points`, with the lat-lon point cloud of each contrail. Each contrail is identified by a `contrail_id` of the form `{satellite}-{region}-{YYYYMMDD-HHMM}-{lat}-{lon}-{hash}`, where the final six characters are a hash of the contrail's pixels, so the same data always give the same IDs.

As with detections, the results are cached. To decompose the masks again, set `force_reprocess_decomposition = True`.

In [ ]:
decomposition_method = "decomposer"  # or "google_api"

prediction_threshold = None  # default for the detection source

contrail_stitching = True  # "decomposer" only

force_reprocess_decomposition = False

In [ ]:
line_segs, point_clouds = meteosat_detector.get_decomposed_contrails(
    method=decomposition_method,
    prediction_threshold=prediction_threshold,
    contrail_stitching=contrail_stitching,
    force_reprocess=force_reprocess_decomposition,
)

In [ ]:
meteosat_detector.contrail_summary = line_segs
meteosat_detector.contrail_point_clouds = point_clouds

#### Decomposed Contrails

This optional plot shows the detection mask beside the decomposed contrails in a zoomed region. The contrails are shown unstitched, so that each decomposed contrail keeps its own colour.

In [ ]:
plot_decomposition = True

if plot_decomposition:
    # Unstitched decomposed contrails, for display only; the pipeline above keeps contrail_stitching
    unstitched_segs, unstitched_points = meteosat_detector.get_decomposed_contrails(
        method="decomposer",
        prediction_threshold=prediction_threshold,
        contrail_stitching=False,
    )
    fig = meteosat_detector.plot_decomposition(
        timestamp=meteosat_detector.timestamp_sequence[0],
        bbox=(0.0, 4.0, 45.0, 49.0),
        contrail_summary=unstitched_segs,
        point_clouds=unstitched_points,
    )
    plt.show()